In [99]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [100]:
# PART 1: SEQ2SEQ (Character-level Translation)

inputs = ["hi", "hello", "bye", "thanks", "yes", "no"]
targets = ["hola", "hola", "adios", "gracias", "si", "no"]

targets = ["\t" + t + "\n" for t in targets]

input_chars = sorted(set("".join(inputs)))
target_chars = sorted(set("".join(targets)))

inp2idx = {c: i+1 for i, c in enumerate(input_chars)}
tar2idx = {c: i+1 for i, c in enumerate(target_chars)}
idx2tar = {i: c for c, i in tar2idx.items()}

num_enc_tokens = len(inp2idx) + 1
num_dec_tokens = len(tar2idx) + 1

In [101]:
# Max lengths for padding
max_enc_len = max(len(x) for x in inputs)
max_dec_len = max(len(x) for x in targets)

# Initialize data tensors
encoder_input = np.zeros((len(inputs), max_enc_len))
decoder_input = np.zeros((len(inputs), max_dec_len))
decoder_target = np.zeros((len(inputs), max_dec_len, num_dec_tokens))

# Encode characters into integer sequences
for i, (inp, tar) in enumerate(zip(inputs, targets)):
    for t, ch in enumerate(inp):
        encoder_input[i, t] = inp2idx[ch]

    for t, ch in enumerate(tar):
        decoder_input[i, t] = tar2idx[ch]

        if t > 0:
            decoder_target[i, t-1, tar2idx[ch]] = 1

latent_dim = 64

In [102]:
# Encoder
enc_inputs = Input(shape=(None,))
enc_embed = Embedding(num_enc_tokens, 16, mask_zero=True)(enc_inputs)
_, h, c = LSTM(latent_dim, return_state=True)(enc_embed)
enc_states = [h, c]

In [103]:
# Decoder
dec_inputs = Input(shape=(None,))
dec_embed_layer = Embedding(num_dec_tokens, 16, mask_zero=True)
dec_embed = dec_embed_layer(dec_inputs)

dec_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
dec_out, _, _ = dec_lstm(dec_embed, initial_state=enc_states)

dec_dense = Dense(num_dec_tokens, activation="softmax")
dec_out = dec_dense(dec_out)

In [104]:
# Model (Encoder + Decoder)
seq2seq_model = Model([enc_inputs, dec_inputs], dec_out)
seq2seq_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

seq2seq_model.fit([encoder_input, decoder_input], decoder_target,
                  epochs=100, batch_size=2, verbose=0)

# Encoder model
encoder_model = Model(enc_inputs, enc_states)

# Decoder model
dec_state_h = Input(shape=(latent_dim,))
dec_state_c = Input(shape=(latent_dim,))
dec_states_inputs = [dec_state_h, dec_state_c]

dec_embed_inf = dec_embed_layer(dec_inputs)
dec_out_inf, h_inf, c_inf = dec_lstm(dec_embed_inf,
                                    initial_state=dec_states_inputs)
dec_out_inf = dec_dense(dec_out_inf)

decoder_model = Model(
    [dec_inputs] + dec_states_inputs,
    [dec_out_inf, h_inf, c_inf]
)

In [105]:
# Prediction Function
def translate(text):
    seq = np.zeros((1, max_enc_len))
    for t, ch in enumerate(text):
        if ch in inp2idx:
            seq[0, t] = inp2idx[ch]

    states = encoder_model.predict(seq, verbose=0)

    target_seq = np.array([[tar2idx["\t"]]])
    result = ""

    while True:
        out, h, c = decoder_model.predict([target_seq] + states, verbose=0)

        idx = np.argmax(out[0, -1, :])
        char = idx2tar.get(idx, "")

        if char == "\n" or len(result) > max_dec_len:
            break

        result += char

        target_seq = np.array([[idx]])
        states = [h, c]

    return result

print("Seq2Seq:")
print("hi ->", translate("hi"))
print("thanks ->", translate("thanks"))
print("\nSeq2Seq Accuracy:", acc)

Seq2Seq:
hi -> hola
thanks -> gracias

Seq2Seq Accuracy: 0.8415300250053406


In [106]:
# PART 2: LSTM NEXT WORD PREDICTION

sentences = [
    "deep learning is powerful",
    "deep learning models are complex",
    "deep learning improves accuracy",
    "deep learning requires large data",
    "machine learning is interesting",
    "machine learning models learn patterns",
    "machine learning is widely used",
    "machine learning improves predictions",
    "artificial intelligence is the future",
    "artificial intelligence solves problems",
    "artificial intelligence powers applications",
    "artificial intelligence is growing fast",
    "i love machine learning",
    "i enjoy learning new skills",
    "i like working with data",
    "i am learning deep learning",
    "lstm is used for sequence prediction",
    "lstm models remember past information",
    "recurrent networks process sequences",
    "sequence models learn order of words",
    "neural networks learn from data",
    "neural networks detect patterns",
    "neural networks improve performance",
    "deep neural networks are powerful",
    "data science is exciting",
    "data science involves statistics",
    "data science uses machine learning",
    "data science helps in decision making",
    "python is easy to learn",
    "python is widely used",
    "python is popular for data science",
    "python supports machine learning libraries",
    "models improve with more data",
    "models learn better with training",
    "good models require good data",
    "better data leads to better models",
    "training data is very important",
    "clean data improves model accuracy",
    "large datasets improve performance",
    "more data helps generalization",
    "learning algorithms find patterns",
    "learning algorithms improve over time",
    "optimization improves model performance",
    "gradient descent updates model weights",
    "loss functions measure error",
    "accuracy measures prediction correctness",
    "overfitting reduces generalization",
    "regularization helps prevent overfitting",
    "practice improves machine learning skills",
    "consistent practice builds strong intuition"
]

# Tokenize text
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

word_index = tokenizer.word_index
vocab_size = len(word_index) + 1

sequences = []
for line in sentences:
    tokens = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(tokens)):
        sequences.append(tokens[:i+1])

# Pad sequences to same length
max_len = max(len(seq) for seq in sequences)
sequences = pad_sequences(sequences, maxlen=max_len, padding="pre")

# Split into X (input) and y (next word)
X = sequences[:, :-1]
y = sequences[:, -1]

# One-hot encode output
y = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

In [107]:
# Model

lstm_model = Sequential([
    Embedding(vocab_size, 10, input_length=max_len-1),
    LSTM(100),
    Dense(vocab_size, activation="softmax")
])

lstm_model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)
lstm_model.fit(X, y, epochs=200, verbose=0)

In [108]:
# Prediction

def predict_next(seed, steps=2):
    for _ in range(steps):
        seq = tokenizer.texts_to_sequences([seed])[0]
        seq = pad_sequences([seq], maxlen=max_len-1, padding="pre")

        pred = lstm_model.predict(seq, verbose=0)
        idx = np.argmax(pred)

        for w, i in word_index.items():
            if i == idx:
                seed += " " + w
                break

    return seed

print("\nLSTM:")
print(predict_next("deep learning"))
print(predict_next("python is"))

loss, acc = lstm_model.evaluate(X, y)
print("\nAccuracy:", acc)


LSTM:
deep learning improves accuracy
python is popular for
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8579 - loss: 0.5626 

Accuracy: 0.8579235076904297
